# CPPWRF: RRTMG FP32 denemesi (Colab)

Bu notebook iki CPPWRF sürümünü **aynı GPU'da** yan yana çalıştırır:

- **base**: ana repo (Xassemblianist/CPPWRF, commit `c9c95c5`), RRTMG FP32 derlemede bile `double` hesaplar.
- **exp**: `SamurayBalgami/CPPWRF` dalı `exp/rrtmg-fp32`, RRTMG FP32 derlemede `float` hesaplar.

Ayrıntılar: `docs/RRTMG_FP32_EXPERIMENT.md`.

**Önce:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **GPU** (T4, L4 veya A100). Sonra hücreleri sırayla çalıştır.
Derleme ücretsiz T4'te (2 çekirdek) uzun sürer (her sürüm için 15-30 dk).

In [ ]:
%%bash
# 1) GPU ve araçlar
nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv
apt-get -qq update > /dev/null && apt-get -qq install -y libnetcdf-dev ninja-build > /dev/null && echo "apt ok"
pip -q install netCDF4 && echo "pip ok"
cmake --version | head -1; nvcc --version | tail -1; g++ --version | head -1; nproc

In [ ]:
%%bash
# 2) Kaynaklar: base = ana repo c9c95c5, exp = fork dalı (c9c95c5 üstüne RRTMG FP32)
cd /content && rm -rf base exp
git clone -q https://github.com/Xassemblianist/CPPWRF.git base && git -C base checkout -q c9c95c5
git clone -q -b exp/rrtmg-fp32 https://github.com/SamurayBalgami/CPPWRF.git exp
git -C base log -1 --format='base %h %s'; git -C exp log -1 --format='exp  %h %s'

In [ ]:
%%bash
# 3) N3 vakasının verisi (WRF girdileri, tablolar, WRF real*8 referansı; 266 MB)
mkdir -p /content/data && cd /content/data
[ -n "$(find . -maxdepth 4 -type d -name run_N3h_r8)" ] || {
  curl -sfL --retry 3 -o n3.tar https://github.com/Xassemblianist/cppwrf-data/releases/download/data-v1/n3_data.tar.part000 &&
  tar xf n3.tar --no-same-owner --no-same-permissions && rm -f n3.tar; }
find /content/data -maxdepth 4 -type d -name run_N3h_r8

In [ ]:
%%bash
# 4) FP32 derlemeleri (base ve exp, aynı ayarlar)
ARCH=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
for c in base exp; do
  cd /content/$c
  t0=$SECONDS
  cmake -S . -B build32 -G Ninja -DCMAKE_BUILD_TYPE=Release -DCPPWRF_ENABLE_CUDA=ON -DCPPWRF_DOUBLE=OFF -DCMAKE_CUDA_ARCHITECTURES=$ARCH > cfg32.log 2>&1 || { echo "$c: cmake FAILED"; tail -30 cfg32.log; continue; }
  ninja -C build32 -j $(nproc) cppwrf > build32.log 2>&1
  echo "$c: build rc=$? ($((SECONDS - t0)) s)"; grep -m10 -E 'error' build32.log
done

In [ ]:
%%bash
# 5) Hız: N3 1 saat (radt 1), önce base sonra exp, iki tur
export CPPWRF_WRF_RUNS=$(dirname $(find /content/data -maxdepth 4 -type d -name run_N3h_r8 | head -1))
mkdir -p /content/out
for i in 1 2; do for c in base exp; do
  cd /content/$c
  /usr/bin/time -f "wall %e s" build32/cppwrf configs/real_n3.ini --set output.dir=/content/out/${c}_1h_$i --set output.format=wrfout > /content/out/${c}_1h_$i.log 2>&1
  echo "== $c tur $i: $(grep -oE 'stepping [0-9.]+ s' /content/out/${c}_1h_$i.log | paste -sd' ') $(tail -1 /content/out/${c}_1h_$i.log)"
done; done

In [ ]:
# 6) Hız özeti
import re, glob
def wall(f): return float(re.findall(r"wall ([0-9.]+) s", open(f).read())[-1])
def step(f): return [float(x) for x in re.findall(r"stepping ([0-9.]+) s", open(f).read())]
for i in (1, 2):
    b, e = f"/content/out/base_1h_{i}.log", f"/content/out/exp_1h_{i}.log"
    sb, se = step(b), step(e)
    print(f"tur {i}: toplam {wall(b):.1f} s -> {wall(e):.1f} s ({wall(b)/wall(e):.2f}x) | "
          f"d01 {sb[0]:.1f} -> {se[0]:.1f} ({sb[0]/se[0]:.2f}x) | d02 {sb[1]:.1f} -> {se[1]:.1f} ({sb[1]/se[1]:.2f}x)")

## İsteğe bağlı: doğruluk (6 saat, gündüz dahil)

Referans: GPU FP64 derlemesi (WRF real*8 ile bit-bit aynı). base FP32 ve exp FP32 6 saat koşar, ikisi de FP64'e karşı
ölçülür. FP64 derlemesi ve 6 saatlik koşular T4'te uzun sürer.

In [ ]:
%%bash
# 7) FP64 referans derlemesi (sadece base; FP64'te iki sürüm bit-bit aynı)
ARCH=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
cd /content/base && cmake -S . -B build64 -G Ninja -DCMAKE_BUILD_TYPE=Release -DCPPWRF_ENABLE_CUDA=ON -DCPPWRF_DOUBLE=ON   -DCMAKE_CUDA_ARCHITECTURES=$ARCH > cfg64.log 2>&1 && ninja -C build64 -j $(nproc) cppwrf > build64.log 2>&1; echo "rc=$?"

In [ ]:
%%bash
# 8) 6 saatlik koşular: FP64 referans, base FP32, exp FP32 (saatlik çıktı)
export CPPWRF_WRF_RUNS=$(dirname $(find /content/data -maxdepth 4 -type d -name run_N3h_r8 | head -1))
L6="--set output.format=wrfout --set run.run_time=21600 --set output.out_interval=60 --set d02.out_interval=60 --set output.diag_interval=60 --set d02.diag_interval=60"
run() { cd /content/$1; /usr/bin/time -f "wall %e s" $2/cppwrf configs/real_n3.ini --set output.dir=/content/out/$3 $L6 > /content/out/$3.log 2>&1
        echo "== $3: $(tail -1 /content/out/$3.log)"; }
run base build64 r64_6h; run base build32 b32_6h; run exp build32 e32_6h

In [ ]:
%%bash
# 9) Doğruluk istatistikleri (repo'nun kendi aracı)
T=/content/base/tools/verify/fp32_accuracy.py; O=/content/out/acc; rm -rf $O; mkdir -p $O
for d in d01 d02; do for t in b32 e32; do
  python3 $T compare /content/out/${t}_6h /content/out/r64_6h --domain $d --pair $t --out $O/${t}_$d.jsonl > /dev/null
done; done; ls $O

In [ ]:
# 10) RMSE tablosu: FP64'e göre hata, exp/base oranı (1'in altı = exp daha yakın)
import json
V = ["T2","Q2","U10","V10","PSFC","TSK","SWDOWN","HFX","LH","PBLH","RAIN","T","QVAPOR","U","V","W","PH","QCLOUD","QRAIN","QICE","QSNOW","QGRAUP"]
def L(t, d): return {(x["var"], round(x["hours"])): x["rmse"] for x in map(json.loads, open(f"/content/out/acc/{t}_{d}.jsonl")) if "rmse" in x}
for d in ("d01", "d02"):
    b, e = L("b32", d), L("e32", d)
    for h in (1, 3, 6):
        rs = sorted(e[k] / b[k] for k in b if k[1] == h and k in e and b[k] > 0 and k[0] in V)
        print(f"{d} {h} h: medyan oran {rs[len(rs)//2]:.2f} (min {rs[0]:.2f}, max {rs[-1]:.2f})")
    print()
    print(f"{d} 6 h  | base | exp | oran")
    for v in V:
        k = (v, 6)
        if k in b and k in e and b[k] > 0: print(f"  {v:7s} {b[k]:.3g} | {e[k]:.3g} | {e[k]/b[k]:.2f}")
    print()